# Notebook 01 of 7 — Getting Started + Providers

*Portfolio Intelligence Engine — User Guide Series.*
[Series README](README.md) · [Story Bible](STORY_BIBLE.md) · Filed under
epic [#1352](https://github.com/prajoria/OpenBB/issues/1352).

---

## Where we are in Sam's story

This is where we meet the system. I've been trading for 14 months on hunches and I'm up 4% while SPY is up 22% over the same window. I don't know if my process is a process or a habit. Someone pointed me at this fork of OpenBB and said 'it will tell you things about your book you don't want to hear.' Let's see.

By the end of this notebook we will be able to answer one question:

> *What does this system actually give me access to, and how do I run it without paying anyone?*


## 0. Before we run anything

The whole series lives in an isolated Python environment called
`.venv_portfolio`. That's deliberate — the fork has a parallel lane
(techtrade) that shares its own venv, and mixing them causes silent
version conflicts. If the cell below halts, follow the printed setup
command exactly, then come back.

*The code cell below asserts we're in the right interpreter and creates
the shared state directory the later notebooks will write into.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] environment sanity — assert .venv_portfolio is active; STATE dir created; friendly halt with setup command if not


## 1. The `obb` object

Everything in the platform hangs off one Python object called `obb`. Two
things live on it:

- **Extensions** — domain areas (`equity`, `portfolio_intel`, `backtest`,
  and so on). Each extension exposes commands you call like a normal
  Python function.
- **Providers** — data sources (`fmp_cached`, `yfinance`, and others).
  When you call an extension command that fetches data, the platform
  routes the request to one of the providers underneath.

The mental model I use: an extension is a *question* ("give me this
company's key metrics"), a provider is *whoever answers* ("here's what
FMP says"). Same question, potentially many answerers.

*The code cell below prints the list of extensions loaded in this venv
and the count of registered fetchers per provider.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] list extensions on obb; print count of fetchers in fmp_cached + yfinance; no output truncation surprises


## 2. Free-tier reality

Two provider rules for this series:

1. **`fmp_cached` first.** It's a caching wrapper around FMP's free
   tier plus a growing library of endpoints we recorded ourselves. If
   the endpoint I need is in `fmp_cached`, that's the one I use.
2. **`yfinance` for the gaps.** Yahoo Finance's public JSON covers a
   lot, doesn't require a key, and always works. When `fmp_cached`
   doesn't have an endpoint (or the free-tier one returns 402), the
   fetcher falls back here.

There is *no* third provider in this series. Every cell you run works
against those two, plus the checked-in snapshots I introduce in §4
below. If you have a paid FMP key, cells will run faster; if you don't,
they still run.

*The code cell below picks one endpoint and shows which provider
actually served it.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] quote MSFT via fmp_cached (default) and via yfinance (explicit); print provider used + row shape for each


## 3. The 20 fetchers that will actually matter

Not all ~160 fetchers matter for what we're doing. Here are the ones
that come back in later notebooks — teased now, so nothing surprises you
later.

**Prices + basic company data (NB02, NB03):**
`EquityQuote`, `EquityHistorical`, `EquityInfo`, `EquityPeers`.

**Fundamentals (NB02):**
`KeyMetricsTtm`, `FinancialRatios`, `FinancialScores`, `OwnerEarnings`,
`EnterpriseValues`, `IncomeStatement`, `BalanceSheet`, `CashFlowStatement`.

**Ratings + calendars (NB02, NB04):**
`Grades`, `PriceTargetConsensus`, `CalendarEarnings`, `HistoricalDividends`.

**Structure + ownership (NB03, NB04):**
`EtfHoldings`, `InstitutionalOwnership`, `InsiderTrading`, `GovernmentTrades`.

**Sector snapshots (NB03):**
`SectorPerformanceSnapshot`.

*The code cell below shows each of these fetchers responding to a
one-line call, with just enough output to prove it works.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] call each of the 20 named fetchers with minimal args; render a compact success/shape table (no full data dumps)


## 4. The offline-snapshot pattern

Two things I care about — the **options chain + implied-vol term
structure** for a name, and the **bond ladder** inside a bond ETF —
don't have a free live JSON API I can hit without a subscription.

Rather than block on that, the fork ships a small framework called
`scrape_record` that lets us **record** a page once (via Playwright) and
then **replay** it from a JSON file checked into the repo. Three
fetchers use this pattern today:

- `YFinanceRecordedOptionsChainsFetcher` — full options chain for a name
- `YFinanceAtmIvTermStructureFetcher` — the ATM implied-vol curve by expiry
- `YFinanceBondLadderFetcher` — bond positions + roll-ups inside a bond ETF

They read from `openbb_platform/tools/scrape_record/snapshots/`. If a
symbol isn't recorded yet, the fetcher raises `EmptyDataError` and tells
you the exact command to record it: `scrape-record record <name>
--symbol <SYM>`. No live-scraping surprises at query time.

*The code cell below reads the bond snapshot for BND — one of the
positions in our through-line basket — and prints the top holdings +
portfolio-average YTM.*

I introduce the full record-replay CLI in NB07.

In [ ]:
# [CODE PLACEHOLDER — Phase B] load YFinanceBondLadderFetcher for BND from the checked-in snapshot; print etf_name, portfolio_avg_ytm, top-5 holdings table


## 5. The through-line basket

From NB03 onward we work against one fixed basket:

| Ticker | Weight |
|--------|--------|
| MSFT   | 12% |
| NVDA   | 10% |
| GOOGL  | 8% |
| AAPL   | 8% |
| AMD    | 6% |
| QQQ    | 15% |
| VTI    | 20% |
| VNQ    | 8% |
| BND    | 10% |
| GLD    | 3% |

Weights are round-number synthetic. This is not anyone's real portfolio.

Look at that list. You probably see what I saw: ten different things,
five of them mega-cap tech, and three ETFs that "diversify" it. It
*looks* diversified. Wait for NB03.

*The code cell below writes this basket to
`.notebook_state/basket.json` so every later notebook loads the exact
same 10 lines.*

In [ ]:
# [CODE PLACEHOLDER — Phase B] write the locked 10-position basket to .notebook_state/basket.json; print confirmation with row count + total weight


---

## What is NOT in this notebook

- **Real-time price feeds / order books.** Everything here is snapshot- or bar-based.
- **Alternative data (satellite, credit-card, dark-pool).** Free-tier only; no paid keys.
- **MCP client demos.** The fork ships an MCP server; how to *drive* it from an outside agent is a separate guide. NB07 has a pointer.

## Preview of NB02

The plumbing works. But before I look at all 10 positions together, I need a repeatable answer to 'what do I actually think of one name?' In NB02 we take MSFT — the largest single position in the basket — and put it through the 7-phase Analysis pipeline end-to-end. Each phase answers one question. The composite hands us a decision label with a staged entry protocol, not a hot take.
